In [0]:
import mlflow
import mlflow.spark
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor, GBTRegressor, LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

In [0]:
#load table and define features
df = spark.read.format("delta") \
    .load("/Volumes/workspace/default/supply_chain_data/delta/features") 

feature_cols = [
    "Product_Price", "order_month", "order_dayofweek", "order_week",
    "shipping_delay", "Late_delivery_risk",
    "qty_lag_7", "qty_lag_14", "qty_lag_30", "qty_rolling_28d_avg",
    "Delivery_Status_idx", "Market_idx", "Department_Name_idx",
    "Category_Name_idx", "Shipping_Mode_idx", "Type_idx"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

print(f"Loaded: {df.count()} rows × {len(df.columns)} cols")

Loaded: 179019 rows × 60 cols


In [0]:
# Training / testing, model evaluation and models to choose
train, test = df.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train.count()} rows | Test: {test.count()} rows")
def evaluate(predictions, metric):
    return RegressionEvaluator(
        labelCol="Order_Item_Quantity",
        predictionCol="prediction",
        metricName=metric
    ).evaluate(predictions)

models = {
    "LinearRegression": LinearRegression(
        labelCol="Order_Item_Quantity",
        featuresCol="features",
        maxIter=20
    ),
    "RandomForest": RandomForestRegressor(
        labelCol="Order_Item_Quantity",
        featuresCol="features",
        numTrees=100,
        maxDepth=6,
        maxBins=100,
        seed=42
    ),
    "GBT": GBTRegressor(
        labelCol="Order_Item_Quantity",
        featuresCol="features",
        maxIter=50,
        maxDepth=5,
        maxBins=100,
        seed=42
    )
}

Train: 143206 rows | Test: 35813 rows


In [0]:
#ML flow 
mlflow.set_experiment("/supply_chain_forecasting")

best_rmse    = float("inf")
best_run_id  = None
best_model   = None
results      = []

for model_name, model in models.items():
    print(f"\nTraining {model_name}")

    with mlflow.start_run(run_name=model_name) as run:

        pipeline = Pipeline(stages=[assembler, model])
        fitted   = pipeline.fit(train)
        preds    = fitted.transform(test)

        rmse = evaluate(preds, "rmse")
        r2   = evaluate(preds, "r2")
        mae  = evaluate(preds, "mae")

        # Log to MLflow
        mlflow.log_param("model_type",   model_name)
        mlflow.log_param("num_features", len(feature_cols))
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2",   r2)
        mlflow.log_metric("mae",  mae)
        mlflow.spark.log_model(fitted, artifact_path="model", dfs_tmpdir="/Volumes/workspace/default/supply_chain_data")

        results.append({
            "model":model_name,
            "rmse":round(rmse, 4),
            "r2":round(r2,   4),
            "mae":round(mae,  4),
            "run_id":run.info.run_id
        })

        print(f"RMSE: {rmse:.4f} | R²: {r2:.4f} | MAE: {mae:.4f}")

        if rmse < best_rmse:
            best_rmse   = rmse
            best_run_id = run.info.run_id
            best_model  = model_name


Training LinearRegression


2026/04/27 17:13:01 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/04/27 17:13:03 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-b7896158-1dec-490d-b8a9-45/tmpfpmt6yix/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/04/27 17:13:03 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


RMSE: 1.0882 | R²: 0.4404 | MAE: 0.7352

Training RandomForest


2026/04/27 17:13:54 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/04/27 17:13:57 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-b7896158-1dec-490d-b8a9-45/tmpftmj2i43/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/04/27 17:13:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


RMSE: 1.0676 | R²: 0.4614 | MAE: 0.6840

Training GBT


2026/04/27 17:15:13 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/04/27 17:15:16 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-b7896158-1dec-490d-b8a9-45/tmpltsh3ns7/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/04/27 17:15:16 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


RMSE: 1.0714 | R²: 0.4575 | MAE: 0.6926


In [0]:
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
print(f"{'Model':<20} {'RMSE':>8} {'R²':>8} {'MAE':>8}")
print("-"*60)
for r in sorted(results, key=lambda x: x["rmse"]):
    flag = "BEST" if r["model"] == best_model else ""
    print(f"{r['model']:<20} {r['rmse']:>8} {r['r2']:>8} {r['mae']:>8}{flag}")
print("="*60)


MODEL COMPARISON
Model                    RMSE       R²      MAE
------------------------------------------------------------
RandomForest           1.0676   0.4614    0.684BEST
GBT                    1.0714   0.4575   0.6926
LinearRegression       1.0882   0.4404   0.7352
